# I.34 精确教学核验

边界顺序A-C-D-B，对角线BC。以下六格由标准库compile/exec实跑，不是Jupyter内核复现。

## 1 前提与长度
坐标是现代欧氏模型，并不认证古代尺规操作。

In [1]:
from geometry import make_certificate, verify_certificate
from cases import EXAMPLE, positives, negatives
c = make_certificate(EXAMPLE)
print(c['opposite_sides_squared'])

{'AB': '16', 'CD': '16', 'AC': '10', 'BD': '10'}


## 2 角
角用归一化(点积,绝对叉积)保存，避免浮点近似。

In [2]:
print(c['alternate_angles'])
print(c['opposite_angles'])

{'ABC': ['1/2', '1/2'], 'BCD': ['1/2', '1/2'], 'ACB': ['1/3', '2/3'], 'CBD': ['1/3', '2/3']}
{'BAC': ['1/4', '3/4'], 'CDB': ['1/4', '3/4'], 'ABD': ['-1/4', '3/4'], 'ACD': ['-1/4', '3/4']}


## 3 分割
检查两三角形对应边、对角线异侧及面积。

In [3]:
print(c['triangles'])
print(c['area'])

{'ABC_squared': ['16', '18', '10'], 'DCB_squared': ['16', '18', '10'], 'ABC_twice_area': '12', 'DCB_twice_area': '12', 'BC_side_A': '12', 'BC_side_D': '-12'}
{'parallelogram': '12', 'ABC': '6', 'DCB': '6'}


## 4 反例
只保留一对平行仍不足。

In [4]:
d = {'A':[0,0], 'B':[4,0], 'C':[1,3], 'D':[6,3]}
try:
    make_certificate(d)
except ValueError as e:
    print('拒绝非平行四边形:', str(e))
else:
    raise RuntimeError('unexpected acceptance')

拒绝非平行四边形: both opposite pairs must be parallel


## 5 有限样例族
正例并非覆盖所有实数坐标；负例直接检查几何入口。

In [5]:
n = 0
for d in positives():
    if not verify_certificate(d, make_certificate(d)):
        raise RuntimeError('failed')
    n += 1
print('精确正例:', n)
m = 0
for d in negatives():
    try:
        make_certificate(d)
    except ValueError:
        m += 1
    else:
        raise RuntimeError('negative accepted')
print('直接拒绝负例:', m)

精确正例: 576
直接拒绝负例: 9


## 6 证书攻击
布尔不是数值证书，平移后的输入需新证书。

In [6]:
import copy
bad = copy.deepcopy(c)
bad['area']['ABC'] = True
print('布尔替换通过:', verify_certificate(EXAMPLE, bad))
shifted = {k:[v[0]+2,v[1]-1] for k,v in EXAMPLE.items()}
print('旧证书移绑通过:', verify_certificate(shifted, c))
print('有限运行不代替一般证明；此Notebook为compile/exec执行，非kernel。')

布尔替换通过: False
旧证书移绑通过: False
有限运行不代替一般证明；此Notebook为compile/exec执行，非kernel。
